# 🌍 Notebook 1 — Pipeline de Building V12

> Construction complète des datasets depuis les sources brutes jusqu'à `dataset_final_v12_couche1.csv`.
> Final V12 : **8 400 lignes × 741 colonnes** (240 pays, 1990-2024, 8 clusters climatiques).

## Pipeline en 12 étapes

| Version | Contenu | Cibles ajoutées |
|---|---|---|
| V1-V2 | Base FAO + WB + WorldClim BIO 1-19 | 6 yields agrégés |
| V3 | NOAA climate + OWID disasters + UCDP + USGS | catastrophes |
| V4 | 39 WB + BIO 2-19 + features dérivées | — |
| V5 | NASA POWER + WHO PM2.5 + forêt OWID | soil_moisture, forest |
| V6 | 36 cultures spécifiques + CRU TS | cultures par espèce |
| V7 | Berkeley Earth 1743-2020 | anomalies multi-décennales |
| **V8 honnête** | **Imputation features + clustering KMeans** | — |
| **V9** | **Couche 1 : élevage + énergie + émissions** | 19 nouvelles cibles |
| **V10** | **Cultures spécifiques cibles + religion Pew + meat per type** | 36 cibles cultures |
| **V11** | **FAO EcoCrop suitability scores** | — (features) |
| **V12** | **Suitability adoucie + indices stress climatique** | — (features) |

> Note : pour seulement charger le dataset final, aller directement à la section 8.

## 1. Imports & configuration

In [ ]:
import os, sys, io, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

DATA_DIR = 'data/cleaned'
RAW_DIR  = 'data/raw'
print(f'Datasets cleaned présents : {len([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])}')

## 2. Mapping noms de pays → codes ISO

Standardisation : noms français FAO, noms anglais WB/OWID, codes alpha-3 FAO → ISO alpha-2 commun.

In [ ]:
from build_dataset import custom_mappings, get_english_iso, get_iso_from_alpha3

print(f'Mappings FR custom : {len(custom_mappings)} pays')
print(f'  "United States" → {get_english_iso("United States")}')
print(f'  "USA" → {get_iso_from_alpha3("USA")}')

## 3. Téléchargement initial (V1-V7)

Tous les datasets externes téléchargés via scripts dédiés.

In [ ]:
# Décommenter pour télécharger tous les datasets externes (~30 min)
# !python download_extra.py        # NOAA indices + OWID catastrophes
# !python download_more.py         # 39 WB indicators
# !python temporalize_geo.py       # Séismes/centrales par année
# !python fetch_nasa_power.py      # NASA POWER (~15 min)
# !python fetch_berkeley_earth.py  # Berkeley Earth (~10 min)
# !python fetch_more_datasets.py   # WHO + GFW + extras
# !python fetch_hansen.py          # Forêt via OWID
# !python add_worldclim_bio.py     # BIO 1-19 sur bbox pays
print('Scripts disponibles. Décommenter pour relancer.')

## 4. Build V1-V7 (base + enrichissements)

```bash
python build_dataset_v2.py    # V2 base
python enrich_v2.py           # V2 → V3 (NOAA + UCDP)
python enrich_v3.py           # V3 → V4 (39 WB + BIO 2-19)
python enrich_v4.py           # V4 → V5 (NASA POWER + WHO)
python enrich_v5.py           # V5 → V6 (cultures spé + CRU)
python enrich_v6.py           # V6 → V7 (Berkeley Earth)
```

In [ ]:
# État des datasets V1-V7
for v in ['v2','v3','v4','v5','v6','v7']:
    f = f'{DATA_DIR}/dataset_final_{v}.csv'
    if os.path.exists(f):
        sz = os.path.getsize(f) / 1e6
        print(f'  ✓ {f}  ({sz:.1f} MB)')
    else:
        print(f'  ✗ {f} absent')

## 5. Imputation honnête V7 → V8

Étape clé : on impute **uniquement les features explicatives**, jamais les cibles ni leurs sources.

### Règles d'imputation
- **Cibles** (`target_*`) et **sources brutes** : intouchables (NaN conservés)
- **Features dynamiques** : interpolation linéaire par pays (limite=5 ans), puis ffill/bfill
- **Features statiques manquantes** : médiane par (cluster × année)

### Clustering
- KMeans 8 clusters sur 15 features environnementales statiques
- Servira en feature additionnelle (Global+Cluster)

In [ ]:
# Imputation + clustering (~30s)
if not os.path.exists(f'{DATA_DIR}/dataset_final_v8_honest.csv'):
    print('Lancement: !python impute_honest.py')
else:
    print(f'✓ dataset_final_v8_honest.csv déjà présent')

## 6. Enrichissements Couche 1 (V8 → V9 → V10 → V11 → V12)

Spécifique au domaine **Planète** (environnement + agriculture + élevage).

### V9 — Élevage + Énergie + Émissions
- FAO `production_animaux` : lait, viandes (bovine/poulet/ovin/porc), œufs
- OWID : émissions CO2/CH4/N2O annuelles par pays
- OWID : énergies (solaire, éolien, hydro, charbon, pétrole, gaz)
- OWID : aquaculture, accès eau potable, marine protected

### V10 — Cultures spécifiques + Religion + Meat per type
- **36 cibles `target_yield_X`** (au lieu d'agrégats faibles) : soja, colza, tomate, pomme, banane...
- **Religion Pew 2010** : % muslim/christian/hindu/buddhist par pays (statique)
- **OWID Meat per type** : beef, pig, poultry, sheep/goat, milk, eggs consumption

### V11 — EcoCrop Suitability
- Téléchargement **FAO EcoCrop database** (2568 espèces)
- Pour 39 cultures principales : T_opt, T_abs, P_opt, P_abs, killing temp, latitude range
- Score 0-1 par (pays, année, culture)

### V12 — Suitability adoucie + Stress climatique
- Suitability EcoCrop **sans zéros stricts** (frost = pénalisation continue)
- **Heat stress index** : (T - 25°C) clipped
- **Frost risk index** : (-T_min) clipped
- **Aridity index** : P / PET (Thornthwaite)
- **Growing season index** v2
- **Continentalité** : amplitude T

In [ ]:
# Build Couche 1 V9-V12 (~5 min)
print('Pour rebuild Couche 1 :')
print('  !python couche1_planete/enrich_couche1.py        # V8 → V9')
print('  !python couche1_planete/fetch_religion_meat.py   # téléchargement extras')
print('  !python couche1_planete/enrich_couche1_v2.py     # V9 → V10')
print('  !python couche1_planete/compute_suitability.py   # V10 → V11')
print('  !python couche1_planete/enrich_v12.py            # V11 → V12')

for v in ['v8_honest','v9_couche1','v10_couche1','v11_couche1','v12_couche1']:
    f = f'{DATA_DIR}/dataset_final_{v}.csv'
    if os.path.exists(f):
        sz = os.path.getsize(f) / 1e6
        print(f'  ✓ {f}  ({sz:.1f} MB)')
    else:
        print(f'  ✗ {f} absent')

## 7. Enrichissement Couche 2 (le Sang)

In [ ]:
# Couche 2 utilise dataset_final_v8_honest.csv directement
# Pas d'enrichissement spécifique nécessaire — les cibles démographiques
# (Birth_Rate, Death_Rate, Fertility, Child_Mort) sont déjà dans V8.
print('Couche 2 charge directement dataset_final_v8_honest.csv')
print('  → 10 cibles démo/santé/catastrophes humaines')

## 8. Chargement du dataset Couche 1 V12

In [ ]:
# Charge V12 si disponible, sinon fallback
for path, version in [(f'{DATA_DIR}/dataset_final_v12_couche1.csv', 'V12'),
                       (f'{DATA_DIR}/dataset_final_v11_couche1.csv', 'V11'),
                       (f'{DATA_DIR}/dataset_final_v10_couche1.csv', 'V10'),
                       (f'{DATA_DIR}/dataset_final_v9_couche1.csv',  'V9'),
                       (f'{DATA_DIR}/dataset_final_v8_honest.csv',   'V8')]:
    if os.path.exists(path):
        df = pd.read_csv(path, low_memory=False)
        print(f'✓ Chargé : {path} ({version})')
        break

df = df.dropna(subset=['ISO']).copy()
df['ISO'] = df['ISO'].astype(str)
df['Annee'] = df['Annee'].astype(int)
if 'cluster' in df.columns:
    df['cluster'] = df['cluster'].astype(int)

print(f'Shape: {df.shape}')
print(f'Pays: {df["ISO"].nunique()} | Années: {df["Annee"].min()}–{df["Annee"].max()}')
if 'cluster' in df.columns:
    print(f'Clusters: {df["cluster"].nunique()}')

## 9. Vérification qualité — cibles disponibles

In [ ]:
targets = [c for c in df.columns if c.startswith('target_')]
print(f'Cibles ({len(targets)}) — taux de non-null :\n')
# Grouper par préfixe
from collections import defaultdict
groups = defaultdict(list)
for t in targets:
    if 'yield_' in t: groups['Agriculture/Élevage'].append(t)
    elif any(k in t for k in ['stress','degradation','thermal','moisture','water_access']): groups['Environnement'].append(t)
    elif any(k in t for k in ['forest','tree','marine']): groups['Écologie'].append(t)
    elif any(k in t for k in ['co2','methane','n2o']): groups['Émissions'].append(t)
    elif any(k in t for k in ['solar','wind','hydro','coal','oil','gas']): groups['Énergie'].append(t)
    elif any(k in t for k in ['birth','death','mortality','fertility','migration','growth','life_exp']): groups['Démographie'].append(t)
    elif any(k in t for k in ['disaster']): groups['Catastrophes'].append(t)
    elif any(k in t for k in ['livestock','milk','carcass','eggs','aquaculture']): groups['Élevage'].append(t)
    else: groups['Autres'].append(t)

for g, ts in groups.items():
    print(f'\n━━ {g} ({len(ts)} cibles) ━━')
    for t in sorted(ts)[:30]:
        n = df[t].notna().sum()
        pct = n/len(df)*100
        print(f'  {t:40s} {n:6,d} ({pct:4.0f}%)')

## 10. NaN — avant vs après imputation V8

In [ ]:
nan_pct_total = df.isna().sum().sum() / (df.shape[0]*df.shape[1]) * 100
print(f'NaN total V12 : {nan_pct_total:.1f}%')

# Features imputables uniquement (hors cibles)
target_sources = ['yield_cereals_kgha','yield_oilcrops_kgha','yield_pulses_kgha','yield_roots_kgha',
                  'yield_fruits_kgha','yield_vegetables_kgha','Water_Withdrawal_pct','Bilan_sols_kgha',
                  'T_anomaly','Child_Mort','Life_Exp','Pop_Growth','Birth_Rate','Death_Rate',
                  'Net_Migration','disaster_deaths','disaster_affected','stunting_pct',
                  'Fertility_Rate','nasa_gwetroot','forest_share_pct','tree_cover_loss_ha']
untouchable = set(targets) | set(target_sources)
for src in target_sources:
    untouchable |= {c for c in df.columns if c.startswith(src + '_')}

imputable = [c for c in df.columns if c not in untouchable and c not in ('ISO','Annee')
             and df[c].dtype in ('float64','int64')]
nan_in_imputable = df[imputable].isna().sum().sum() / (df.shape[0]*len(imputable)) * 100
print(f'NaN dans features imputables : {nan_in_imputable:.2f}%  (≈0 attendu)')

## 11. Profil des 8 clusters climatiques

In [ ]:
cluster_profile = pd.read_csv(f'{DATA_DIR}/country_clusters.csv')
profile_stats = cluster_profile.groupby('cluster').agg(
    n_pays=('ISO', 'nunique'),
    T_mean=('temp_mean', 'mean'),
    P_mean=('precip_mean', 'mean'),
    Lat=('latitude', 'mean'),
    Elev=('elevation', 'mean'),
).round(1)
print('Profil moyen par cluster :\n')
print(profile_stats.to_string())
print('\nExemples par cluster :')
for c in sorted(cluster_profile['cluster'].unique()):
    members = cluster_profile[cluster_profile['cluster']==c]['ISO'].head(8).tolist()
    n = len(cluster_profile[cluster_profile["cluster"]==c])
    print(f'  Cluster {c} ({n} pays) : {members}')

## 12. Familles de variables V12

In [ ]:
GROUPS = {
    'Climat dynamique':      [c for c in df.columns if any(k in c for k in ['T_annual','P_annual','T_anomaly','P_anomaly','nasa_t2m','nasa_prec','be_t_anom'])],
    'Climat WorldClim':      [c for c in df.columns if c.startswith('bio') or c in ('temp_mean','precip_mean','wind_speed_mean')],
    'Stress climatique':     [c for c in df.columns if any(k in c for k in ['heat_stress','frost_risk','aridity','pet_annual','continentality','growing_season','climate_zone'])],
    'Suitability EcoCrop':   [c for c in df.columns if c.startswith('suit_')],
    'Sol & humidité':        [c for c in df.columns if any(k in c for k in ['clay','silt','sand','soil_pH','organic_carbon','gwet','Bilan_sols'])],
    'Agriculture':           [c for c in df.columns if any(k in c for k in ['Engrais','Pesticides','Terres','Bio_','Irrigation']) or (c.startswith('yield_') and 'kgha' in c)],
    'Élevage':               [c for c in df.columns if any(k in c for k in ['livestock','milk','meat_','eggs','aquaculture','dairy'])],
    'Religion (Pew)':        [c for c in df.columns if c.startswith('share_') and ('muslim' in c or 'christian' in c or 'hindu' in c or 'buddhist' in c or 'folk' in c or 'unaffiliated' in c or 'jew' in c)],
    'Géographie statique':   [c for c in df.columns if any(k in c for k in ['latitude','longitude','elevation','slope','dist_to_'])],
    'Démographie':           [c for c in df.columns if any(k in c for k in ['Population','GDP','HDI','Child_Mort','Life_Exp','Birth_Rate','Fertility','Pop_Growth','Urban_pct'])],
    'Catastrophes':          [c for c in df.columns if 'disaster' in c or 'earthquake' in c or 'volcan' in c],
    'Énergie':               [c for c in df.columns if any(k in c for k in ['elec_','Energy','Renew','Electricity','solar_','wind_','hydro_','coal_','oil_','gas_'])],
    'Indices climatiques':   [c for c in df.columns if any(k in c for k in ['enso','nao','amo','pdo','soi','ao_lag','co2_ppm'])],
    'Forêt':                 [c for c in df.columns if any(k in c for k in ['forest_','tree_cover','deforest'])],
    'Conflits':              [c for c in df.columns if 'conflict' in c],
    'Émissions':             [c for c in df.columns if any(k in c for k in ['co2_','methane','n2o'])],
    'Cluster':               ['cluster'],
}
print(f"{'Famille':<25s} {'#cols':>6s}")
print('─' * 35)
for g, cols in GROUPS.items():
    cols = [c for c in cols if c in df.columns]
    print(f'  {g:<25s} {len(cols):>5d}')

## 13. Récapitulatif final V12

In [ ]:
print('═' * 60)
print('PIPELINE V12 — RÉCAPITULATIF')
print('═' * 60)
print(f'Shape         : {df.shape}')
print(f'Pays uniques  : {df["ISO"].nunique()}')
print(f'Année min/max : {df["Annee"].min()}–{df["Annee"].max()}')
if 'cluster' in df.columns:
    print(f'Clusters      : {df["cluster"].nunique()} (climat-éco)')
print(f'Cibles ML     : {len(targets)}')
print(f'NaN total     : {nan_pct_total:.1f}%')
print(f'Sources       : FAO + WB + OWID + WorldClim + NASA POWER + Berkeley Earth')
print(f'                + EcoCrop + Pew Research + WHO + USGS + NOAA + UCDP')
print()
print('→ Voir notebook 2 (visualization) et 3 (modelisation_complete).')
print('→ Ou par couche : couche1_planete/modelisation_couche1.ipynb, couche2_sang/modelisation_couche2.ipynb')